# Week 4 Workshop — Notebook 1: Dimensionality Reduction and Neural Manifolds

**Theoretical Neuroscience — University of Copenhagen**

In this notebook you will:
1. Simulate a population of neurons with structured, low-dimensional activity
2. Visualise the population response as a **raster plot**
3. Apply **Principal Component Analysis (PCA)** and **Non-negative Matrix Factorisation (NMF)** to the spike-count matrix
4. Plot the **neural manifold trajectory** using the first three components of each method
5. Compare the representations learned by the two methods

---

## 0. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from sklearn.decomposition import PCA, NMF
from mpl_toolkits.mplot3d import Axes3D

rng = np.random.default_rng(seed=42)

print('All imports successful.')

---
## 1. Simulate a population with structured dynamics

We generate a population of **N neurons** whose firing rates are driven by a small number of **latent variables** (sinusoids at different frequencies), plus Poisson noise. This mimics the situation in real cortical recordings, where population activity lies on a low-dimensional manifold.

The firing rate of neuron $i$ at time $t$ is:
$$\lambda_i(t) = r_0 + \sum_{k=1}^{K} w_{ik} \, s_k(t)$$
where $s_k(t)$ are the latent signals and $w_{ik}$ are random mixing weights.

In [ ]:
# ── Simulation parameters ──────────────────────────────────────────────────────
N_neurons  = 80          # number of neurons
T          = 2.0         # total time (s)
dt         = 0.005       # time bin width (s)  → 5 ms bins
n_latent   = 3           # number of latent signals (true manifold dimension)
r0         = 5.0         # baseline firing rate (Hz)
mod_depth  = 8.0         # modulation depth (Hz)

t = np.arange(0, T, dt)   # time axis
n_bins = len(t)

# ── Latent signals: three sinusoids at different frequencies ──────────────────
freqs = [1.5, 3.0, 6.0]   # Hz
phases = rng.uniform(0, 2*np.pi, n_latent)
S = np.array([np.sin(2*np.pi*f*t + phi)
              for f, phi in zip(freqs, phases)])  # shape: (n_latent, n_bins)

# ── Random mixing weights ──────────────────────────────────────────────────────
W = rng.standard_normal((N_neurons, n_latent))     # shape: (N_neurons, n_latent)
W /= np.linalg.norm(W, axis=1, keepdims=True)      # normalise rows

# ── Firing rate matrix ─────────────────────────────────────────────────────────
Lambda = r0 + mod_depth * (W @ S)                  # shape: (N_neurons, n_bins)
Lambda = np.clip(Lambda, 0.5, None)                # keep rates positive

# ── Inhomogeneous Poisson spike trains ─────────────────────────────────────────
spikes = rng.poisson(Lambda * dt)                  # spike counts per bin

print(f'Population: {N_neurons} neurons, {n_bins} time bins ({T} s at {1/dt:.0f} Hz)')
print(f'Mean firing rate: {spikes.sum() / (N_neurons * T):.1f} Hz')

---
## 2. Raster plot

A **raster plot** shows the times at which each neuron fired a spike. Each row is a neuron; each dot is a spike.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

for i in range(N_neurons):
    spike_times = t[spikes[i, :] > 0]
    ax.scatter(spike_times, np.full_like(spike_times, i),
               marker='|', s=12, color='steelblue', linewidths=0.6, alpha=0.8)

ax.set_xlabel('Time (s)', fontsize=13)
ax.set_ylabel('Neuron index', fontsize=13)
ax.set_title('Population raster plot', fontsize=14)
ax.set_xlim([0, T])
ax.set_ylim([-1, N_neurons])
plt.tight_layout()
plt.show()

**Questions:**
- Can you see any structure in the raster? Do neurons appear to be correlated?
- How would the raster change if you increased `mod_depth` or changed `n_latent`?

---
## 3. Smooth the spike counts (square-kernel)

Before dimensionality reduction, we smooth the spike counts with a boxcar filter to estimate firing rates.

In [ ]:
def smooth_boxcar(X, half_width=5):
    """Row-wise boxcar smoothing."""
    kernel = np.ones(2 * half_width + 1) / (2 * half_width + 1)
    return np.array([np.convolve(row, kernel, mode='same') for row in X])

X_smooth = smooth_boxcar(spikes.astype(float), half_width=4) / dt  # convert to Hz

# X_smooth has shape (N_neurons, n_bins)
# For sklearn, we need shape (n_bins, N_neurons) — samples × features
X_T = X_smooth.T   # shape: (n_bins, N_neurons)

print('Data matrix shape (time × neurons):', X_T.shape)

---
## 4. Principal Component Analysis (PCA)

PCA finds orthogonal directions of maximum variance in the data. The first principal component (PC1) captures the most variance, PC2 the second most, and so on.

$$\mathbf{Z}_{\text{PCA}} = (\mathbf{X} - \bar{\mathbf{X}}) \mathbf{V}$$

where $\mathbf{V}$ contains the eigenvectors of the covariance matrix.

In [ ]:
n_components = 3

pca = PCA(n_components=n_components)
Z_pca = pca.fit_transform(X_T)   # shape: (n_bins, 3)

explained = pca.explained_variance_ratio_
print('Variance explained by PC1, PC2, PC3:')
for k, ev in enumerate(explained):
    print(f'  PC{k+1}: {ev*100:.1f}%')
print(f'  Total (3 PCs): {explained.sum()*100:.1f}%')

# ── Scree plot ─────────────────────────────────────────────────────────────────
pca_full = PCA().fit(X_T)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(np.arange(1, len(pca_full.explained_variance_ratio_)+1),
        np.cumsum(pca_full.explained_variance_ratio_)*100, 'o-', color='steelblue')
ax.axvline(n_latent, color='tomato', linestyle='--', label=f'True rank ({n_latent})')
ax.set_xlabel('Number of PCs', fontsize=12)
ax.set_ylabel('Cumulative variance explained (%)', fontsize=12)
ax.set_title('PCA scree plot', fontsize=13)
ax.legend()
ax.set_xlim([1, 20])
plt.tight_layout()
plt.show()

**Questions:**
- How many PCs are needed to capture ~90% of the variance? Does this match the true number of latent signals (`n_latent = 3`)?
- What does the elbow in the scree plot indicate?

---
## 5. Non-negative Matrix Factorisation (NMF)

NMF factorises the data matrix $\mathbf{X} \approx \mathbf{H} \mathbf{W}$ subject to $\mathbf{H}, \mathbf{W} \geq 0$. This non-negativity constraint tends to produce **parts-based**, more interpretable representations — each component corresponds to a localised pattern of activity rather than a global oscillation.

In [ ]:
# NMF requires non-negative input
X_nn = X_T - X_T.min()   # shift to be non-negative

nmf = NMF(n_components=n_components, init='nndsvd', max_iter=500, random_state=0)
Z_nmf = nmf.fit_transform(X_nn)   # shape: (n_bins, 3)

recon_err = nmf.reconstruction_err_
print(f'NMF reconstruction error: {recon_err:.2f}')
print('NMF component shapes:', Z_nmf.shape)

---
## 6. Manifold trajectories in 3D

We now plot the **neural manifold trajectory**: the path traced by the population state vector through the 3D subspace defined by the first three components. Each point along the trajectory corresponds to one time bin.

In [ ]:
# ── Color the trajectory by time ───────────────────────────────────────────────
colors = plt.cm.viridis(np.linspace(0, 1, n_bins))

fig = plt.figure(figsize=(15, 6))

for col_idx, (Z, label) in enumerate([(Z_pca, 'PCA'), (Z_nmf, 'NMF')]):
    ax = fig.add_subplot(1, 2, col_idx + 1, projection='3d')

    # Plot trajectory as a coloured line
    for i in range(n_bins - 1):
        ax.plot(Z[i:i+2, 0], Z[i:i+2, 1], Z[i:i+2, 2],
                color=colors[i], linewidth=1.0, alpha=0.85)

    # Mark start and end
    ax.scatter(*Z[0],  color='lime',   s=60, zorder=5, label='Start')
    ax.scatter(*Z[-1], color='tomato', s=60, zorder=5, label='End')

    ax.set_xlabel(f'{label}1', fontsize=10)
    ax.set_ylabel(f'{label}2', fontsize=10)
    ax.set_zlabel(f'{label}3', fontsize=10)
    ax.set_title(f'Neural manifold — {label}', fontsize=13)
    ax.legend(fontsize=9)

# Shared colorbar for time
sm = plt.cm.ScalarMappable(cmap='viridis', norm=plt.Normalize(vmin=0, vmax=T))
sm.set_array([])
cbar = fig.colorbar(sm, ax=fig.axes, shrink=0.5, pad=0.02)
cbar.set_label('Time (s)', fontsize=11)

plt.suptitle('Population manifold trajectory (colour = time)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

**Questions:**
- Describe the shape of the manifold trajectory. What does it reflect about the latent signals used to generate the data?
- How do the PCA and NMF trajectories differ qualitatively? Which components are easier to interpret?
- Would you expect the manifold to be more or less regular if you increased Poisson noise (lower `mod_depth`)?

---
## 7. Plot the components over time

It can also be helpful to inspect each component as a time series to understand what each dimension captures.

In [ ]:
fig, axes = plt.subplots(n_components, 2, figsize=(13, 7), sharex=True)

for k in range(n_components):
    for col_idx, (Z, label, color) in enumerate([
            (Z_pca, 'PCA', 'steelblue'),
            (Z_nmf, 'NMF', 'darkorange')]):
        axes[k, col_idx].plot(t, Z[:, k], color=color, linewidth=1.2)
        axes[k, col_idx].set_ylabel(f'{label}{k+1}', fontsize=11)
        if k == 0:
            axes[k, col_idx].set_title(label, fontsize=13)
        if k == n_components - 1:
            axes[k, col_idx].set_xlabel('Time (s)', fontsize=11)

plt.suptitle('Latent components over time', fontsize=14)
plt.tight_layout()
plt.show()

**Questions:**
- Compare the PCA and NMF components. Are they oscillatory? What frequencies do you recognise?
- PCA components can be negative; NMF components cannot. Does this change your interpretation of what each component represents?

---
## 8. Exercise: change the latent dimensionality

> **Try it yourself:** Go back to Section 1 and change `n_latent` to 1, 2, or 5. Re-run all cells.
>
> - How does the scree plot change?
> - How does the shape of the 3D manifold trajectory change?
> - At what point does the three-component representation fail to capture all the structure?

---
*End of Notebook 1*